# 02 — Préparation des données et construction des variables

## Objectif

Cette étape vise à transformer les données brutes éCO2mix en un jeu de données
supervisé exploitable pour le Machine Learning.

Les travaux réalisés portent sur :

- le filtrage du périmètre retenu ;
- le traitement des doublons et anomalies identifiés lors de l'exploration ;
- la construction de la variable cible à J+1 ;
- la création de variables temporelles et historiques ;
- le contrôle des valeurs manquantes générées par les décalages temporels ;
- la vérification de l'absence de fuite d'information ;
- l'export d'un dataset préparé pour la modélisation.

Cette étape constitue une preuve directe de la compétence C5.2.1 du référentiel RNCP.

In [43]:
from pathlib import Path

import numpy as np
import pandas as pd

In [44]:
PROJECT_ROOT = Path.cwd().parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "eco2mix-regional-cons-def.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "nouvelle_aquitaine_ml.parquet"

In [45]:
df = pd.read_csv(
    RAW_PATH,
    sep=";",
    na_values=["ND", "N/A", ""],
    low_memory=False
)

In [46]:
df["Date - Heure"] = pd.to_datetime(
    df["Date - Heure"],
    utc=True,
    errors="coerce"
)

In [47]:
df = df[
    (df["Région"] == "Nouvelle-Aquitaine")
    & (df["Date - Heure"].dt.year >= 2019)
].copy()

In [48]:
df = df.sort_values("Date - Heure").reset_index(drop=True)

In [49]:
if "Column 30" in df.columns:
    df = df.drop(columns=["Column 30"])

In [50]:
df = df.drop_duplicates(
    subset=["Région", "Date - Heure"],
    keep="last"
).copy()

## Traitement des doublons temporels

Les doublons identifiés lors de l'exploration apparaissent principalement autour
des changements d'heure de fin mars et de fin octobre.

L'analyse a montré que ces lignes présentent les mêmes valeurs de consommation
et de production pour un même timestamp.

Une seule occurrence est donc conservée afin d'éviter de dupliquer artificiellement
une observation dans la série temporelle.

In [51]:
df.loc[df["Consommation (MW)"] <= 0]

,Code INSEE région,Région,Nature,Date,Heure,Date - Heure,Consommation (MW),Thermique (MW),Nucléaire (MW),Eolien (MW),...,TCO Nucléaire (%),TCH Nucléaire (%),TCO Eolien (%),TCH Eolien (%),TCO Solaire (%),TCH Solaire (%),TCO Hydraulique (%),TCH Hydraulique (%),TCO Bioénergies (%),TCH Bioénergies (%)


In [52]:
df["delta_t"] = df["Date - Heure"].diff()

df["delta_t"].value_counts().head(10)

delta_t
0 days 00:30:00    131398
0 days 01:30:00         7
Name: count, dtype: int64

In [53]:
HORIZON = 48

df["target_consumption_t_plus_48"] = (
    df["Consommation (MW)"]
    .shift(-HORIZON)
)

In [54]:
consumption_by_time = (
    df.set_index("Date - Heure")["Consommation (MW)"]
)

lags_hours = {
    "30min": pd.Timedelta(minutes=30),
    "1h": pd.Timedelta(hours=1),
    "3h": pd.Timedelta(hours=3),
    "6h": pd.Timedelta(hours=6),
    "24h": pd.Timedelta(hours=24),
    "48h": pd.Timedelta(hours=48),
    "7d": pd.Timedelta(days=7),
}

for name, delta in lags_hours.items():
    df[f"consumption_lag_{name}"] = (
        consumption_by_time
        .reindex(df["Date - Heure"] - delta)
        .to_numpy()
    )

In [55]:
df["consumption_roll_mean_6"] = (
    df["Consommation (MW)"]
    .shift(1)
    .rolling(window=6)
    .mean()
)

df["consumption_roll_mean_48"] = (
    df["Consommation (MW)"]
    .shift(1)
    .rolling(window=48)
    .mean()
)

df["consumption_roll_mean_336"] = (
    df["Consommation (MW)"]
    .shift(1)
    .rolling(window=336)
    .mean()
)

df["consumption_roll_std_48"] = (
    df["Consommation (MW)"]
    .shift(1)
    .rolling(window=48)
    .std()
)

In [56]:
dt = df["Date - Heure"]

df["hour"] = dt.dt.hour
df["minute"] = dt.dt.minute
df["day_of_week"] = dt.dt.dayofweek
df["month"] = dt.dt.month
df["day_of_year"] = dt.dt.dayofyear
df["is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)

In [57]:
df["half_hour_slot"] = (
    df["hour"] * 2
    + (df["minute"] // 30)
)

In [58]:
df["hour_sin"] = np.sin(2 * np.pi * df["half_hour_slot"] / 48)
df["hour_cos"] = np.cos(2 * np.pi * df["half_hour_slot"] / 48)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

In [59]:
df["consumption_diff_1"] = (
    df["Consommation (MW)"]
    - df["consumption_lag_30min"]
)

df["consumption_diff_48"] = (
    df["Consommation (MW)"]
    - df["consumption_lag_24h"]
)

In [60]:
target_lookup = (
    df[["Date - Heure", "Consommation (MW)"]]
    .copy()
)

target_lookup["Date - Heure"] = (
    target_lookup["Date - Heure"] - pd.Timedelta(hours=24)
)

target_lookup = target_lookup.rename(
    columns={"Consommation (MW)": "target_consumption_t_plus_24h"}
)

df = df.merge(
    target_lookup,
    on="Date - Heure",
    how="left"
)

In [63]:
feature_cols = [
    col for col in df.columns
    if col.startswith("consumption_lag_")
    or col.startswith("consumption_roll_")
    or col.startswith("consumption_diff_")
    or col in [
        "hour",
        "minute",
        "day_of_week",
        "month",
        "day_of_year",
        "is_weekend",
        "half_hour_slot",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "month_sin",
        "month_cos",
    ]
]

df[feature_cols + ["target_consumption_t_plus_24h"]].isna().sum().sort_values(
    ascending=False
)

consumption_lag_7d               350
consumption_roll_mean_336        336
consumption_lag_48h              110
consumption_lag_24h               62
target_consumption_t_plus_24h     62
consumption_diff_48               62
consumption_roll_std_48           48
consumption_roll_mean_48          48
consumption_lag_6h                26
consumption_lag_3h                20
consumption_lag_1h                16
consumption_diff_1                 8
consumption_lag_30min              8
consumption_roll_mean_6            6
hour                               0
day_of_year                        0
minute                             0
day_of_week                        0
month                              0
hour_sin                           0
half_hour_slot                     0
is_weekend                         0
hour_cos                           0
month_sin                          0
dow_cos                            0
dow_sin                            0
month_cos                          0
d

In [64]:
ml_df = df[
    ["Date - Heure"]
    + feature_cols
    + ["target_consumption_t_plus_24h"]
    + ["Consommation (MW)"]
].dropna().copy()

In [70]:
ml_df.shape

(130924, 29)

In [71]:
ml_df["consumption_t"] = ml_df["Consommation (MW)"]

In [73]:
ml_df["target_consumption_t_plus_24h"].describe()

count    130924.000000
mean       4781.373973
std        1089.508631
min        2055.000000
25%        3984.000000
50%        4588.000000
75%        5447.000000
max        9496.000000
Name: target_consumption_t_plus_24h, dtype: float64

In [74]:
check = pd.DataFrame({
    "timestamp": df["Date - Heure"],
    "consumption_now": df["Consommation (MW)"],
    "target": df["target_consumption_t_plus_24h"]
})

check.head(60)

,timestamp,consumption_now,target
0,2019-01-01 00:00:00+00:00,6391.0,5949.0
1,2019-01-01 00:30:00+00:00,6543.0,6091.0
2,2019-01-01 01:00:00+00:00,6341.0,5879.0
3,2019-01-01 01:30:00+00:00,6384.0,5949.0
4,2019-01-01 02:00:00+00:00,6119.0,5703.0
5,2019-01-01 02:30:00+00:00,5935.0,5538.0
6,2019-01-01 03:00:00+00:00,5718.0,5422.0
7,2019-01-01 03:30:00+00:00,5615.0,5410.0
8,2019-01-01 04:00:00+00:00,5547.0,5435.0
9,2019-01-01 04:30:00+00:00,5582.0,5589.0


In [75]:
ml_df.drop(columns="Consommation (MW)", inplace=True)

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

ml_df.to_parquet(
    PROCESSED_PATH,
    index=False
)

In [77]:
print("Dataset final :", ml_df.shape)
print("Début :", ml_df["Date - Heure"].min())
print("Fin :", ml_df["Date - Heure"].max())

Dataset final : (130924, 29)
Début : 2019-01-08 00:00:00+00:00
Fin : 2026-06-29 21:30:00+00:00


## Conclusion

Le jeu de données brut a été transformé en un jeu supervisé adapté à une tâche
de prévision de consommation électrique à J+1.

La variable cible correspond à la consommation observée 48 pas de temps après
l'instant de référence, soit 24 heures plus tard.

Les variables explicatives construites décrivent :

- l'historique récent de la consommation ;
- la consommation au même créneau les jours précédents ;
- des statistiques glissantes ;
- les variations récentes ;
- le contexte calendaire ;
- la cyclicité journalière, hebdomadaire et annuelle.

Les variables ont été construites uniquement à partir d'informations disponibles
au moment de la prédiction afin d'éviter toute fuite de données.

Le dataset obtenu constitue la base utilisée lors de l'étape suivante pour
sélectionner les variables, construire les baselines et entraîner les modèles.